# Week 3 — Loss Landscape Topology & Edge of Stability

> Visualizing the loss surface around a converged minimum with filter-normalized random directions, and detecting the Edge-of-Stability regime via Hessian power iteration.

---

## 1. Theory

### 1.1 The loss landscape problem

For a network with parameters $\theta \in \mathbb{R}^p$ (often $p \in 10^6, 10^9$), the loss surface $\mathcal{L}(\theta)$ is a scalar function of an absurd-dimensional input. **Random 2-D slices** are the standard visualization technique, but they have a subtle pitfall.

### 1.2 Filter normalization (Li et al., 2018)

Naive random directions $\delta \sim \mathcal{N}(0, I)$ ignore the scale-invariance of ReLU networks: any rescaling of weights in one layer can be exactly absorbed into the next. A direction sampled with norm independent of $\theta^*$ effectively asks an arbitrary question. **Filter normalization** fixes this by rescaling each filter:

$$\hat{\delta}_{i, j} \;=\; \frac{\delta_{i, j}}{\|\delta_{i, j}\|_F} \cdot \|\theta_{i, j}\|_F$$

so the perturbation has the same per-filter scale as the trained network.

### 1.3 Loss-surface slice

For two filter-normalized directions $\hat\delta, \hat\eta$, the 2-D slice is

$$\mathcal{L}(\alpha, \beta) \;=\; \mathcal{L}\bigl(\theta^* + \alpha \hat\delta + \beta \hat\eta\bigr)$$

Plotting $\mathcal{L}(\alpha, \beta)$ as a contour (or 3-D surface) reveals basin shape, saddle structure, and the steepness of curvature in the principal directions.

### 1.4 Hessian curvature & Edge of Stability

The largest Hessian eigenvalue $\lambda_{\max}(\theta)$ controls local quadratic curvature:

$$\mathcal{L}(\theta + \delta) \;\approx\; \mathcal{L}(\theta) + \langle \nabla \mathcal{L}, \delta\rangle + \tfrac{1}{2}\,\lambda_{\max}\,\langle \delta, v_{\max}\rangle^2 + O(\|\delta\|^3)$$

Cohen et al. (2021) showed that gradient descent on neural nets typically operates **at the threshold**

$$\lambda_{\max}(\theta_t) \;\geq\; \frac{2}{\eta}$$

after a "progressive sharpening" phase. In this regime the loss is **non-monotone** but globally descending — it oscillates with period $\approx \pi$ steps.

### 1.5 Power iteration for $\lambda_{\max}$

Computing the Hessian is intractable ($p^2$ entries). Power iteration on Hessian-vector products $Hv$ (via double backward through `torch.autograd.grad`) gives $\lambda_{\max}$ in $O(\text{cost of one backward})$ per iteration, typically converging in 20–50 steps.

### 1.6 Lanczos for extremes

Lanczos builds an orthonormal Krylov basis $\{q_1, Hq_1, H^2 q_1, \ldots\}$, projecting $H$ to a small tridiagonal $T_m$. The extreme eigenvalues of $T_m$ converge to those of $H$. This gives both $\lambda_{\min}$ and $\lambda_{\max}$ in one pass.


In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.utils import set_global_seed
from src.optimization import (
    TrajectoryStore,
    compute_loss_grid,
    detect_edge_of_stability,
    generate_filter_normalized_direction,
    hessian_top_eigenvalue,
    lanczos_extrema,
    project_trajectory,
)

set_global_seed(0)
print("TensorLens — Week 3 notebook loaded")


## 2. Training a small MLP at the edge of stability

We deliberately pick a learning rate $\eta$ that puts the model into the EOS regime within ~80 steps.

In [ ]:
class TinyMLP(nn.Module):
    def __init__(self, d_in: int = 8, d_hidden: int = 32, d_out: int = 1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, d_hidden),
            nn.Tanh(),
            nn.Linear(d_hidden, d_hidden),
            nn.Tanh(),
            nn.Linear(d_hidden, d_out),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


torch.manual_seed(42)
N_TRAIN = 200
D_IN = 8
X = torch.randn(N_TRAIN, D_IN)
true_w = torch.randn(D_IN, 1) * 0.5
Y = X @ true_w + 0.1 * torch.randn(N_TRAIN, 1)

model = TinyMLP(d_in=D_IN, d_hidden=32)


def loss_on_batch(m: nn.Module) -> torch.Tensor:
    return ((m(X) - Y) ** 2).mean()


LR = 0.10  # large LR to push toward EOS
N_STEPS = 90
HESSIAN_INTERVAL = 5

store = TrajectoryStore()
losses: list[float] = []
lam_max_trace: list[float] = []

opt = torch.optim.SGD(model.parameters(), lr=LR)
for step in range(N_STEPS):
    opt.zero_grad(set_to_none=True)
    loss = loss_on_batch(model)
    loss.backward()
    opt.step()
    losses.append(float(loss.detach()))
    if step % HESSIAN_INTERVAL == 0:
        lam, _ = hessian_top_eigenvalue(
            lambda: loss_on_batch(model),
            model,
            n_iter=20,
            seed=0,
        )
        lam_max_trace.append(lam)
        store.capture(model, step, metrics={"loss": losses[-1], "lambda_max": lam})

print(f"Trained {N_STEPS} steps, captured {len(store)} snapshots")
print(f"λ_max trace head: {[round(x, 3) for x in lam_max_trace[:5]]}")
print(f"λ_max trace tail: {[round(x, 3) for x in lam_max_trace[-5:]]}")
print(f"2/η = {2 / LR:.3f}")


## 3. Edge of Stability detection

We test for the EOS regime by checking whether $\lambda_\max$ ever holds a plateau above $2/\eta$ accompanied by oscillating loss.

In [ ]:
# Resample lambda_max to match loss length via simple zero-order hold
lam_full = []
for step in range(N_STEPS):
    idx = min(step // HESSIAN_INTERVAL, len(lam_max_trace) - 1)
    lam_full.append(lam_max_trace[idx])

report = detect_edge_of_stability(
    losses,
    lam_full,
    learning_rate=LR,
    plateau_steps=3,
    threshold_slack=0.95,
    oscillation_threshold=1e-5,
)
print(report)


In [ ]:
fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(y=losses, mode="lines", name="loss"))
fig.add_trace(
    go.Scatter(
        y=lam_full,
        mode="lines",
        name="λ_max",
        line=dict(color="firebrick"),
    ),
    secondary_y=True,
)
fig.add_hline(
    y=2 / LR,
    line=dict(color="firebrick", dash="dash"),
    annotation_text="2/η",
    annotation_position="top right",
    secondary_y=True,
)
if report.onset_step >= 0:
    fig.add_vline(
        x=report.onset_step,
        line=dict(color="purple", dash="dot"),
        annotation_text="EOS onset",
    )
fig.update_yaxes(title="loss", secondary_y=False)
fig.update_yaxes(title="λ_max", secondary_y=True)
fig.update_layout(height=420, width=1100, title="Loss curve vs Hessian top eigenvalue")
fig.show()


## 4. Loss landscape slice via filter-normalized directions

Two filter-normalized directions sampled around the final trained weights, evaluated on a fine 2-D grid. The contour plot reveals basin curvature; the surface plot lets us see saddle structure.

In [ ]:
# Save the final state
final_state = {k: v.detach().clone() for k, v in model.state_dict().items()}

dir_a = generate_filter_normalized_direction(model, seed=11)
dir_b = generate_filter_normalized_direction(model, seed=22)

alphas, betas, grid = compute_loss_grid(
    model=model,
    anchor_state=final_state,
    direction_a=dir_a,
    direction_b=dir_b,
    alpha_range=(-1.0, 1.0),
    beta_range=(-1.0, 1.0),
    n_alpha=25,
    n_beta=25,
    loss_fn=lambda m: float(loss_on_batch(m)),
)
print(f"Loss-grid shape: {grid.shape}, range [{grid.min():.4f}, {grid.max():.4f}]")


In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "contour"}, {"type": "surface"}]],
    subplot_titles=("Contour of L(α, β)", "3-D surface of L(α, β)"),
)
fig.add_trace(
    go.Contour(z=grid, x=alphas, y=betas, colorscale="Viridis", contours_coloring="heatmap"),
    row=1, col=1,
)
fig.add_trace(
    go.Surface(z=grid, x=alphas, y=betas, colorscale="Viridis", showscale=False),
    row=1, col=2,
)
fig.update_layout(height=520, width=1200, title="Loss landscape — filter-normalized 2-D slice")
fig.show()


## 5. Projecting the SGD trajectory onto the slice

We project every captured snapshot's parameter delta from the anchor onto $(\hat\delta, \hat\eta)$. Plotting this curve over the contour shows where the optimizer wandered before settling.

In [ ]:
coords = project_trajectory(store.state_dicts(), dir_a, dir_b, anchor=final_state)
coords_np = coords.numpy()

fig = go.Figure()
fig.add_trace(go.Contour(z=grid, x=alphas, y=betas, colorscale="Viridis",
                         contours_coloring="heatmap", showscale=True))
fig.add_trace(go.Scatter(x=coords_np[:, 0], y=coords_np[:, 1], mode="lines+markers",
                         marker=dict(size=6, color=list(range(len(coords_np))), colorscale="Inferno"),
                         line=dict(color="white", width=1),
                         name="trajectory"))
fig.add_trace(go.Scatter(x=[0.0], y=[0.0], mode="markers",
                         marker=dict(size=12, symbol="star", color="red"), name="anchor θ*"))
fig.update_layout(height=520, width=900, title="SGD trajectory projected onto filter-normalized 2-D slice",
                  xaxis_title="α (δ̂)", yaxis_title="β (η̂)")
fig.show()


## 6. Lanczos: both ends of the Hessian spectrum

Lanczos on the final-state model gives both $\lambda_\min$ and $\lambda_\max$ in one pass. The presence of meaningfully negative eigenvalues is the signature of a **saddle** rather than a true minimum.

In [ ]:
lam_min, lam_max = lanczos_extrema(lambda: loss_on_batch(model), model, m=15, seed=0)
print(f"Lanczos λ_min = {lam_min:.4f}")
print(f"Lanczos λ_max = {lam_max:.4f}")
print(f"Condition number κ = {lam_max / max(abs(lam_min), 1e-8):.2f}")
if lam_min < -1e-3:
    print("⚠ Negative curvature direction present — current θ* is saddle-adjacent.")
else:
    print("✓ No significant negative curvature — θ* sits in a quasi-convex basin.")


## 7. Take-aways

1. **Filter normalization is non-negotiable.** Without it, the visualized slice tells you about your random direction, not your loss surface.
2. **EOS is the rule, not the exception.** Modern training with Adam or large-LR SGD virtually always operates at $\lambda_\max \approx 2/\eta$; smooth descent is a sign of small step sizes leaving optimization performance on the table.
3. **Trajectories pierce contours.** Projecting the captured weight history onto the slice exposes whether the optimizer arrived at the basin directly, or whether it skipped across a ridge — the latter is informative about momentum.
4. **Power iteration costs one Hessian-vector product per step.** Practical for in-training monitoring.

### References

* Li, H. et al. (2018). *Visualizing the Loss Landscape of Neural Nets.* NeurIPS.
* Cohen, J. et al. (2021). *Gradient Descent on Neural Networks Typically Occurs at the Edge of Stability.* ICLR.
* Pearlmutter, B. (1994). *Fast Exact Multiplication by the Hessian.* Neural Computation.
* Ghorbani, B., Krishnan, S., Xiao, Y. (2019). *An Investigation into Neural Net Optimization via Hessian Eigenvalue Density.* ICML.
